# Colab bootstrap (Day 18)

Run this cell-by-cell the first time. After that, run the whole notebook at the
start of every session.

The purpose of this notebook is **parity**: the same checks that pass locally
must pass here. If they do not, stop and fix it before running experiments,
because a silent environment difference will show up later as an
uninterpretable result.

In [ ]:
# 1. GPU and machine identity -- record this in the logbook every session
!nvidia-smi || echo 'no GPU: CPU fallbacks apply (see manual Section 23)'
import platform, sys
print(platform.platform()); print(sys.version)

In [ ]:
# 2. Mount Drive. Checkpoints and activation caches must survive a disconnect.
from google.colab import drive
drive.mount('/content/drive')
PROJECT = '/content/drive/MyDrive/unlearning-audit'
import os; os.makedirs(PROJECT, exist_ok=True); print(PROJECT)

In [ ]:
# 3. Clone (first time) or pull (afterwards)
%cd $PROJECT
!git clone <YOUR_REPO_URL> repo 2>/dev/null || (cd repo && git pull)
%cd $PROJECT/repo
!git rev-parse HEAD

In [ ]:
# 4. Install. Colab's preinstalled torch is usually right -- do not fight it.
!pip install -q transformer-lens sae-lens
print('RESTART THE RUNTIME NOW if pip changed numpy or torch, then re-run from cell 1.')

In [ ]:
# 5. Version log -- write it into the run directory, every session
import sys; sys.path.insert(0, '.')
from src.utils.config import library_versions, git_commit
from src.utils.io import write_json
import datetime
v = library_versions(); v['git_commit'] = git_commit()
v['timestamp'] = datetime.datetime.now().isoformat()
write_json(v, 'results/logs/session_versions.json')
v

In [ ]:
# 6. PARITY CHECKS -- all three must pass before any experiment
!python -m pytest tests -q
!python scripts/99_pipeline_smoke_test.py | tail -5
!python scripts/00_token_check.py --config configs/base.yaml

## Memory discipline

Free-tier sessions die from three things, in this order:

1. **Caching every hook point.** Always pass `names_filter`. `src.analysis.activations`
   already does this; do not bypass it.
2. **Holding the reference model in float32 alongside the trained model.** For NPO
   you need both. On a 16GB T4 that is fine for GPT-2 Small and tight for anything
   larger; if you move to Gemma-2-2B, cache the reference log-probabilities once
   instead of keeping the model resident.
3. **Never calling `del` + `torch.cuda.empty_cache()` between sweep runs.**

Write checkpoints to Drive as you go. A sweep that has to restart from step 0
because the session dropped costs a day.